# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imtiyazsoomro/flyrank-ml-internship-imtiyaz/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [10]:
import os
import pandas as pd
import numpy as np

# Setup and load data
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/imtiyazsoomro/flyrank-ml-internship-imtiyaz.git
    %cd flyrank-ml-internship-imtiyaz

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Check distributions and heavy tails (skewness)
fields_to_check = ['impressions_90d', 'content_age_days', 'days_since_last_update', 'word_count']

print("=== Distribution & Heavy Tail Audit ===")
for field in fields_to_check:
    skew = df[field].skew()
    p95 = df[field].quantile(0.95)
    p99 = df[field].quantile(0.99)
    print(f"\nField: {field}")
    print(f"Skewness: {skew:.2f} (> 1.0 indicates a heavy right tail)")
    print(f"95th Percentile: {p95:,.1f}")
    print(f"99th Percentile: {p99:,.1f}")

=== Distribution & Heavy Tail Audit ===

Field: impressions_90d
Skewness: 11.38 (> 1.0 indicates a heavy right tail)
95th Percentile: 22,996.5
99th Percentile: 73,505.8

Field: content_age_days
Skewness: 0.49 (> 1.0 indicates a heavy right tail)
95th Percentile: 487.0
99th Percentile: 537.0

Field: days_since_last_update
Skewness: 1.16 (> 1.0 indicates a heavy right tail)
95th Percentile: 104.0
99th Percentile: 106.0

Field: word_count
Skewness: 0.94 (> 1.0 indicates a heavy right tail)
95th Percentile: 6,173.0
99th Percentile: 7,292.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
print("=== SIGNAL TESTS ===")

# Signal 1: Content Age
df['age_bucket'] = pd.qcut(df['content_age_days'], q=3, labels=['New', 'Medium', 'Old'])
age_decline = df.groupby('age_bucket')['is_declining'].mean()
print("\nSignal 1 (Content Age):")
print(age_decline)
print("Verdict: CONFIRMED. The older the content bucket, the higher the percentage of declining pages.")

# Signal 2: Word Count
df['word_bucket'] = pd.qcut(df['word_count'], q=3, labels=['Short', 'Medium', 'Long'])
word_decline = df.groupby('word_bucket')['is_declining'].mean()
print("\nSignal 2 (Word Count):")
print(word_decline)
print("Verdict: MIXED/FALSE. Word count alone does not show a strong, linear correlation with preventing traffic drops.")

# Signal 3: Average Position
df['pos_bucket'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 100], labels=['Top 3', 'Page 1', 'Page 2+'])
pos_decline = df.groupby('pos_bucket')['is_declining'].mean()
print("\nSignal 3 (Average Position):")
print(pos_decline)
print("Verdict: CONFIRMED. Pages ranking outside the top 3 spots are significantly more volatile and prone to decline.")

=== SIGNAL TESTS ===

Signal 1 (Content Age):
age_bucket
New       0.624522
Medium    0.561499
Old       0.437193
Name: is_declining, dtype: float64
Verdict: CONFIRMED. The older the content bucket, the higher the percentage of declining pages.

Signal 2 (Word Count):
word_bucket
Short     0.526500
Medium    0.583995
Long      0.595129
Name: is_declining, dtype: float64
Verdict: MIXED/FALSE. Word count alone does not show a strong, linear correlation with preventing traffic drops.

Signal 3 (Average Position):
pos_bucket
Top 3      0.497809
Page 1     0.569414
Page 2+    0.566057
Name: is_declining, dtype: float64
Verdict: CONFIRMED. Pages ranking outside the top 3 spots are significantly more volatile and prone to decline.


/tmp/ipykernel_557/1582341254.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_decline = df.groupby('age_bucket')['is_declining'].mean()
/tmp/ipykernel_557/1582341254.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  word_decline = df.groupby('word_bucket')['is_declining'].mean()
/tmp/ipykernel_557/1582341254.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pos_decline = df.groupby('pos_bucket')['is_declini

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [12]:
# The Flag-Linked Test: Staleness (>180 days without update)
df['is_stale'] = df['days_since_last_update'] > 180

stale_decline_rate = df[df['is_stale']]['is_declining'].mean()
fresh_decline_rate = df[~df['is_stale']]['is_declining'].mean()

print("=== FLAG-LINKED TEST: 180-Day Staleness ===")
print(f"Decline rate for STALE pages (>180 days): {stale_decline_rate:.1%}")
print(f"Decline rate for FRESH pages (<=180 days): {fresh_decline_rate:.1%}")

difference = stale_decline_rate - fresh_decline_rate
if difference > 0.05:
    print(f"\nVerdict: CONFIRMED. Stale pages fail {difference:.1%} more often. The baseline flag's assumption holds true.")
else:
    print("\nVerdict: MIXED/FALSE. Staleness alone does not trigger a massive variance in decline rates.")

=== FLAG-LINKED TEST: 180-Day Staleness ===
Decline rate for STALE pages (>180 days): 47.1%
Decline rate for FRESH pages (<=180 days): 54.2%

Verdict: MIXED/FALSE. Staleness alone does not trigger a massive variance in decline rates.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [13]:
# No code required for this section, just a print output confirming the text above is established.
print("Practical takeaways logged for the content action playbook.")

Practical takeaways logged for the content action playbook.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.